# Module 5.1: Next-Token Prediction & Cross-Entropy Loss

Welcome to Phase 2: Training & Alignment! Everything we've built so far mathematically calculates a *prediction*, but a freshly initialized network will just spout random garbage. It needs to learn by penalizing "wrong" predictions and rewarding "right" ones.

In this notebook, we look at the mathematical objective of all LLMs (**Next-Token Prediction**) and the function used to score them (**Cross-Entropy Loss**).

## 1. The Supervised Signal

How do we know if our Decoder is "right"?

In Language Modeling, the "target" is simply the *very next word* in the dataset. We shift our input targets by 1 position into the future!

```mermaid
graph TD
    A[(Dataset: 'Deep learning is fun')] --> B(Input Context)
    A --> C(Target Labels)
    
    B -.->|'Deep'| D1(Model)
    C -.->|Target:| E1{'learning'}
    D1 -->|Compares to| E1
    
    B -.->|'Deep learning'| D2(Model)
    C -.->|Target:| E2{'is'}
    D2 -->|Compares to| E2
```

In [ ]:
import torch

# Assume our Vocabulary consists of just 10 words, numbered 0 to 9.
# Let's say our document sequence is: [2, 5, 8, 1, 9]
sequence = torch.tensor([2, 5, 8, 1, 9])

# We create Inputs and Targets by shifting the sequence by 1!
x_inputs = sequence[:-1]  # Everything except the very last token
y_targets = sequence[1:]  # Everything except the very first token

for i in range(len(x_inputs)):
    # Note: During training, models actually process all contexts in parallel 
    # thanks to the Lower Triangular Mask we established in Module 3!
    print(f"Context: {sequence[:i+1]} ---> Target to predict: {y_targets[i]}")

## 2. Cross-Entropy Loss

Our model outputs a gigantic array of Logits (e.g. 10 numbers for our 10-word vocabulary). We convert this to probabilities using `Softmax`.

If the target word was `word_id = 5`, we want the probability score at index `5` to be exactly `1.0`, and all other indices to be `0.0`. 

**Cross-Entropy Loss** calculates the mathematical "distance" between our predicted probability distribution and the "true" distribution (which is just a 1 at the correct index!). Let's write it from scratch, then use the PyTorch optimized version.

In [ ]:
import torch.nn.functional as F

# 1. Mock the model prediction
# Pretend our model looked at the context `[2]` and spat out 10 raw scores (Logits)
logits = torch.randn(1, 10) 

# 2. Get probabilities
probs = torch.softmax(logits, dim=-1)

# 3. Target label
target = torch.tensor([5]) # The word we WANTED it to predict
print(f"True label id: {target.item()}")
print(f"Model's predicted probability for true label: {probs[0, target.item()]:.4f}")

# -- FROM SCRATCH CROSS ENTROPY --
# CrossEntropy = -log(probability_of_the_correct_class)
negative_log_likelihood = -torch.log(probs[0, target.item()])
print(f"Loss (From Scratch): {negative_log_likelihood.item():.4f}")

# -- PYTORCH BUILT-IN (Usually combined for numerical stability) --
# PyTorch takes the RAW logits, so it can do Softmax and Log safely under the hood!
pytorch_loss = F.cross_entropy(logits, target)
print(f"Loss (PyTorch):      {pytorch_loss.item():.4f}")

## Summary

By predicting the "next token" and calculating the **Cross-Entropy Loss**, we get a single mathematical metric that represents how "surprised" our model was by the true text. Lower Loss = Better Predictions. 

Now that we have a Loss number, we can use Calculus (backpropagation) to walk backward through our architecture and edit the matrices to lower this number. That is exactly what we will do in **Module 5.2: The Training Loop**!